# DP-Fusion prototype

The mechanism itself, one layer at a time. `fusit.trace` decides *which* tokens are private;
this notebook is about what DP-Fusion does with that answer.

The idea in one line: run the model twice — once on the document, once on a redacted copy —
and sample from a blend of the two next-token distributions, with the blend held close enough
to the redacted run that the private tokens provably cannot have moved it far.

    p_out = (1/m) Σᵢ [ λᵢ·p_priv,i + (1-λᵢ)·p_pub ],   λᵢ = max λ s.t. D↔_α(mix ‖ p_pub) ≤ αβᵢ

λ is the whole story. It is not a hyperparameter — it is re-solved at *every* decoding step,
by bisection, as the largest mixing weight the privacy budget still allows. When the private
and public runs agree, λ goes to 1 and you get the private distribution for free; when they
disagree, λ collapses and the output falls back to the redacted view.

Sections 3–5 take one decoding step apart to show that happening. Section 6 runs the real
generator, 7 turns the per-step divergences into an (ε, δ) guarantee, and 8 sweeps the budget.

**Kernel**: `Python (dpfusion-repro)`. **Hardware**: Qwen2.5-7B in fp16 needs ~15 GB, plus
m+1 sequences of activations per step. `MODEL_ID` at the top switches to the 0.5B.

## 1. Setup

In [ ]:
import textwrap, time

import torch

from fusit.dataset import ATTACK_ENTITY_TYPES, TAB_ENTITY_TYPES, get_dataset
from fusit.dp_fusion import (
    build_aligned_tokens,
    build_contexts,
    compute_dp_epsilon,
    compute_epsilon_single_group,
    compute_renyi_divergence_clipped_symmetric,
    dp_fusion_groups_incremental,
    find_lambda,
)
from fusit.dp_fusion.prompting import format_prompt_new_template

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"   # the paper's model; "Qwen/Qwen2.5-0.5B-Instruct" to iterate
DOC_CHARS = 1500                         # truncate the document; full TAB docs run ~5k chars
# Which types get their OWN privacy group. The paper treats all eight as private and only
# scopes its *attack* to PERSON/CODE/DATETIME (Section 5.1) -- two different things. Narrowing
# this narrows what gets its own epsilon, not what stays secret: build_contexts redacts every
# span from PUBLIC either way. More groups means m+1 forward passes per step but a tighter
# per-group epsilon.
GROUPS    = TAB_ENTITY_TYPES             # the paper's setup; ATTACK_ENTITY_TYPES is cheaper

ALPHA = 2.0        # paper: "We use standard alpha = 2, delta = 0.001"
DELTA = 1e-3

# Naming trap. The paper writes the per-group budget as beta and caps the divergence at
# alpha*beta (Algorithm 1 line 6). `dp_fusion_groups_incremental` takes the **cap itself** in
# beta_dict -- section 6 confirms the realised divergence tops out at exactly this number --
# so this is the paper's alpha*beta, the quantity Table 1 and Table 4 are indexed by.
MAX_DIV = 0.10                  # the paper sweeps alpha*beta over 0.01 .. 0.10
BETA = MAX_DIV / ALPHA          # the paper's beta, which is what Theorem 4 wants

print(f"torch {torch.__version__} | cuda {torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"{p.name}, {p.total_memory/1e9:.1f} GB")
print(f"alpha={ALPHA}  alpha*beta={MAX_DIV} (the cap)  ->  paper's beta = {BETA}")


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

t0 = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="cuda:0")
model.eval()
print(f"loaded {MODEL_ID} in {time.perf_counter()-t0:.0f}s | {torch.cuda.memory_allocated()/1e9:.1f} GB")

## 2. A document, and its privacy groups

TAB-ECHR ships hand-annotated spans, so this notebook takes the ground truth as given and
studies the mechanism. A real deployment would put `fusit.trace` or a NER tagger here, and the
paper is explicit that the guarantee only ever covers what the tagger actually marked.

Note the distinction the paper draws and that is easy to lose: *every* annotated span is
private and gets redacted from the public context, but only the types listed in `GROUPS` get
their own λ, their own divergence trace and their own ε. A span of an ungrouped type is never
revealed in any context — strictly stronger protection, just without a per-group accounting.


In [ ]:
tab = get_dataset("tab_echr")
doc = tab.select(n=1, seed=0)[0]
TEXT = doc.text[:DOC_CHARS]
SPANS = [s for s in doc.spans if s.end <= len(TEXT)]

print(f"{doc.doc_id}: {len(TEXT)} chars, {len(SPANS)} private spans")
from collections import Counter
for t, n in Counter(s.entity_type for s in SPANS).most_common():
    print(f"  {t:9s} {n:3d}  {[s.text for s in SPANS if s.entity_type == t][:3]}")

def redact(text, spans, ph="_"):
    out, prev = [], 0
    for s in sorted(spans, key=lambda s: s.start):
        out.append(text[prev:s.start]); out.append(ph * (s.end - s.start)); prev = s.end
    out.append(text[prev:])
    return "".join(out)

print("\n--- what the public run sees ---")
print(textwrap.fill(redact(TEXT, SPANS)[:600], 100))

### The contexts, and the length trap

Every group's token sequence must be the **same length**. The paper pads each redacted span
with an equal number of placeholder tokens "to prevent length-based leakage" — a shorter
public context would announce how much was removed before a single token is generated.

Tokenizing the two prompts independently does *not* give you that, which is the trap
`build_contexts` exists to avoid: it tokenizes the full prompt once and swaps only the ids
inside redacted spans. The cell below shows both paths.

In [ ]:
ctx = build_contexts(tokenizer, TEXT, SPANS, entity_types=GROUPS)

# build_contexts skips requested types with no spans in this document -- a group with nothing
# to reveal costs a forward pass per token and changes nothing. Work from what it actually
# built, not from what was asked for.
GROUPS = [k for k in ctx if k != "PUBLIC"]
print(f"requested {len(TAB_ENTITY_TYPES) if GROUPS else 0} types, built {len(GROUPS)} groups:", GROUPS)
print("token counts:", {k: len(v) for k, v in ctx.items()})
assert len({len(v) for v in ctx.values()}) == 1
print("-> all equal, so nothing leaks through length\n")

for name, toks in ctx.items():
    if name == "PUBLIC":
        continue
    print(f"  {name:9s} reveals {sum(a != b for a, b in zip(ctx['PUBLIC'], toks)):3d} tokens PUBLIC hides")

# the naive path, for contrast: tokenize each string separately
naive_priv = tokenizer(format_prompt_new_template(tokenizer, TEXT, "_"), add_special_tokens=False)["input_ids"]
naive_pub  = tokenizer(format_prompt_new_template(tokenizer, redact(TEXT, SPANS), "_"), add_special_tokens=False)["input_ids"]
print(f"\ntokenized separately: private {len(naive_priv)} vs public {len(naive_pub)} tokens"
      f"  ->  {'ALIGNED' if len(naive_priv) == len(naive_pub) else 'MISALIGNED, guarantee broken'}")

## 3. One decoding step: the two distributions

Everything above was bookkeeping. Here is the actual quantity the mechanism operates on —
the model's next-token distribution with the private tokens visible, and without.

In [ ]:
import torch.nn.functional as Fn

@torch.no_grad()
def next_token_dist(tokens, temperature=1.0):
    """Softmax in float32, for the reason section 5 measures: the divergence between two of
    these is small enough that fp16 reports mostly its own rounding. The library's generation
    loop casts for the same reason."""
    ids = torch.tensor([tokens], device=model.device)
    logits = model(input_ids=ids).logits[0, -1, :].float()
    return Fn.softmax(logits / temperature, dim=-1)

p_pub = next_token_dist(ctx["PUBLIC"])
p_priv = {g: next_token_dist(ctx[g]) for g in GROUPS}

def top(p, k=6):
    v, i = torch.topk(p, k)
    return [(tokenizer.decode([j]), round(float(x), 4)) for x, j in zip(v, i)]

print("p_pub  ", top(p_pub))
for g in GROUPS:
    print(f"p_priv[{g:8s}]", top(p_priv[g]))

## 4. How far apart are they?

The mechanism measures distance with the **symmetric** Rényi divergence,
`D↔_α(p‖q) = max{D_α(p‖q), D_α(q‖p)}` (Definition 5). Symmetric because adjacency is
add/remove: both "this group was added" and "this group was removed" have to be bounded, and
taking the max enforces both at once.

In [ ]:
print(f"alpha = {ALPHA}, divergence cap alpha*beta = {MAX_DIV}\n")
for g in GROUPS:
    d_full = compute_renyi_divergence_clipped_symmetric(p_priv[g], p_pub, ALPHA)
    verdict = "within cap at lambda=1" if d_full <= MAX_DIV else "over cap -> lambda must shrink"
    print(f"  {g:9s} D<->(p_priv || p_pub) = {float(d_full):8.5f}   {verdict}")

## 5. Solving for λ

Theorem 3: the divergence is non-decreasing in λ. That monotonicity is what makes the search
cheap — bisection finds the largest admissible λ in ~14 evaluations instead of a scan.

**Only in float32.** These divergences are small, the vocabulary is ~150k, and fp16 cannot
resolve the differences: on one real step the same quantity measured 0.0477 in fp16 against
0.0084 in fp32, and the fp16 curve came out non-monotone with negative values — which would
put bisection outside its own precondition. `dp_fusion_groups_incremental` therefore softmaxes
in float32 while keeping the weights fp16. The cell below reproduces the comparison, because a
5.7x measurement error on the quantity the ε accounting consumes is worth seeing once.

The paper plots this curve in Figures 6 and 7 and notes the shape is not fixed: CODE came out
roughly logarithmic, DATETIME closer to a power law. Worth *looking* at rather than assuming.

In [ ]:
import torch.nn.functional as Fn

@torch.no_grad()
def raw_logits(tokens):
    return model(input_ids=torch.tensor([tokens], device=model.device)).logits[0, -1, :]

lg_pub, lg_priv = raw_logits(ctx["PUBLIC"]), raw_logits(ctx[GROUPS[0]])
lam_grid = [i / 10 for i in range(11)]

print(f"group {GROUPS[0]}, D<->(mix || p_pub) as lambda sweeps 0 -> 1\n")
print(f"{'lambda':>7} | {'fp32 (used)':>12} | {'fp16':>12}")
print("-" * 38)
curves = {}
for dt, label in ((torch.float32, "fp32"), (torch.float16, "fp16")):
    q = Fn.softmax(lg_pub.to(dt), dim=-1)
    pp = Fn.softmax(lg_priv.to(dt), dim=-1)
    curves[label] = [float(compute_renyi_divergence_clipped_symmetric(l * pp + (1 - l) * q, q, ALPHA))
                     for l in lam_grid]
for l, a, b in zip(lam_grid, curves["fp32"], curves["fp16"]):
    print(f"{l:7.1f} | {a:12.6f} | {b:12.6f}")

for label, vals in curves.items():
    mono = all(y >= x - 1e-9 for x, y in zip(vals, vals[1:]))
    print(f"\n{label}: monotone={mono}, negatives={sum(v < 0 for v in vals)}, max={max(vals):.5f}")

In [ ]:
lambdas, divergences = {}, {}
for g in GROUPS:
    lam, div = find_lambda(p_priv[g], p_pub, alpha=ALPHA, beta=ALPHA*BETA)
    lambdas[g], divergences[g] = lam, div
    bar = "#" * round(40 * lam)
    print(f"  {g:9s} lambda={lam:6.4f}  divergence={div:8.5f} <= {ALPHA*BETA}  {bar}")

assert all(d <= ALPHA*BETA + 1e-9 for d in divergences.values()), "budget violated"
print("\nevery group is inside its budget")

Now mix and sample — Algorithm 1 line 8. The output is the **average** over groups of each
group's mixture, which is where the `(m-1)/m + (1/m)e^(...)` in the accounting comes from:
one group's private distribution only ever gets `1/m` of the weight.

In [ ]:
mixed = sum(lambdas[g] * p_priv[g] + (1 - lambdas[g]) * p_pub for g in GROUPS) / len(GROUPS)

print("p_pub  ", top(p_pub))
print("mixed  ", top(mixed))
print("p_priv ", top(p_priv[GROUPS[0]]), f"  <- group {GROUPS[0]} alone")

moved = float((mixed - p_pub).abs().sum()) / 2
print(f"\ntotal variation between the mixture and the public distribution: {moved:.4f}")
print(f"sampled token: {tokenizer.decode([int(torch.multinomial(mixed, 1))])!r}")

## 6. Generating for real

`dp_fusion_groups_incremental` is the loop: it does the above at every step, KV-cached, with
all m+1 contexts batched into one forward pass. That batching is why the paper can say the
latency is "approximately equivalent to that of a single LLM forward pass" despite needing
m+1 of them.

In [ ]:
# beta_dict takes the divergence cap directly -- see the note in section 1
beta_dict = {g: MAX_DIV for g in GROUPS}

t0 = time.perf_counter()
paraphrase, lam_hist, div_hist = dp_fusion_groups_incremental(
    token_ids_groups={k: list(v) for k, v in ctx.items()},
    beta_dict=beta_dict,
    alpha=ALPHA,
    model=model,
    tokenizer=tokenizer,
    temperature=1.0,
    max_new_tokens=80,
)
elapsed = time.perf_counter() - t0

generated = paraphrase[len(tokenizer.decode(ctx["PUBLIC"], skip_special_tokens=True)):]
print(f"{elapsed:.1f}s for {len(next(iter(lam_hist.values())))} steps, {len(GROUPS)+1} contexts per step\n")
print(textwrap.fill(generated.strip(), 100))

In [ ]:
print(f"{'group':>9} | {'lambda':^23} | {'divergence':^23}")
print(f"{'':>9} | {'mean':>7}{'min':>8}{'max':>8} | {'mean':>7}{'min':>8}{'max':>8}")
print("-" * 62)
for g in GROUPS:
    L, D = lam_hist[g], div_hist[g]
    print(f"{g:>9} | {sum(L)/len(L):7.3f}{min(L):8.3f}{max(L):8.3f} | "
          f"{sum(D)/len(D):7.4f}{min(D):8.4f}{max(D):8.4f}")

over = {g: sum(1 for d in div_hist[g] if d > MAX_DIV + 1e-9) for g in GROUPS}
print(f"\nsteps above the cap {MAX_DIV}: {over}   (must be all zero)")
assert not any(over.values())
assert all(d >= 0 for g in GROUPS for d in div_hist[g]), "negative divergence -- precision problem"
print(f"observed maximum: {max(max(div_hist[g]) for g in GROUPS):.5f}"
      f"  -- confirms beta_dict is the cap, not the paper's beta")

## 7. From divergences to (ε, δ)

Theorem 4 converts the per-step budget into a guarantee over the whole T-token transcript:

    εᵢ = T · (1/(α-1)) · log( (m-1)/m + (1/m)·e^((α-1)·4βᵢ) ) + log(1/δ)/(α-1)

Two things to read off it. ε grows **linearly in T** — a longer paraphrase costs more privacy,
so `max_new_tokens` is a privacy parameter, not just a speed one. And more groups *tighten*
per-group ε, because one group's distribution carries only `1/m` of the mixture.

**Mind which β goes in.** The βᵢ in that formula is the paper's β, but what section 6 logged
are divergences, capped at α·β. The package's two accountants read that differently:
`compute_epsilon_single_group` divides its input by α (so it wants divergences),
`compute_dp_epsilon` does not (so it wants β). Fed the same list they disagree by a factor of
α — 26.91 against 46.91 at α=2, T=100, αβ=0.1. The cell below checks both against Theorem 4
computed directly, rather than trusting either name.

In [ ]:
import math

T, m = len(next(iter(div_hist.values()))), len(GROUPS)

def theorem4(betas, alpha=ALPHA, delta=DELTA, m=m):
    """Eq. 5, summed per step. `betas` are the paper's beta, i.e. divergence / alpha."""
    per_step = sum((1/(alpha-1)) * math.log((m-1)/m + (1/m)*math.exp((alpha-1)*4*b)) for b in betas)
    return per_step + math.log(1/delta)/(alpha-1)

print(f"T = {T} tokens, m = {m} groups, alpha = {ALPHA}, delta = {DELTA}")
print("all three columns are Theorem 4; they differ only in m and in what they are fed\n")
print(f"{'group':>9} {'Thm4 (m=' + str(m) + ')':>13} {'compute_dp_epsilon':>20} {'single_group':>14}")
print(f"{'':>9} {'':>13} {'(m=1, fed beta)':>20} {'(m=1, fed div)':>14}")
print("-" * 60)
for g in GROUPS:
    betas = [d / ALPHA for d in div_hist[g]]          # divergences -> the paper's beta
    t4 = theorem4(betas)
    cdp = compute_dp_epsilon({g: betas}, alpha=ALPHA, delta=DELTA, mode="global")
    e1 = compute_epsilon_single_group(div_hist[g], alpha=ALPHA, delta=DELTA)["empirical"]
    print(f"{g:>9} {t4:13.3f} {cdp:20.3f} {e1:14.3f}")

print("\nThe last two agree because each was handed the input its own convention expects:")
print("  compute_dp_epsilon treats its input as beta, compute_epsilon_single_group divides by alpha.")
print("Hand either one the other's input and the answer moves by a factor of alpha.")
wrong = compute_dp_epsilon({GROUPS[0]: div_hist[GROUPS[0]]}, alpha=ALPHA, delta=DELTA, mode="global")
right = compute_dp_epsilon({GROUPS[0]: [d/ALPHA for d in div_hist[GROUPS[0]]]}, alpha=ALPHA, delta=DELTA, mode="global")
print(f"  e.g. {GROUPS[0]}: fed divergences -> {wrong:.3f}, fed beta -> {right:.3f}")

In [ ]:
# observed vs worst case: Appendix A.7 reports epsilon_data staying below epsilon_theo,
# because the realised divergence usually sits well under the cap
print(f"{'group':>9} {'observed':>10} {'worst case':>12}   (worst case = every step pinned at the cap)")
print("-" * 54)
for g in GROUPS:
    obs = theorem4([d / ALPHA for d in div_hist[g]])
    worst = theorem4([MAX_DIV / ALPHA] * T)
    print(f"{g:>9} {obs:10.3f} {worst:12.3f}   ({100*obs/worst:4.1f}% of the bound)")

print(f"\nepsilon is linear in T -- the same per-step divergences at other lengths:")
for mult in (1, 2, 4):
    e = theorem4([d / ALPHA for d in div_hist[GROUPS[0]]] * mult)
    print(f"  T = {T*mult:4d}  ->  epsilon = {e:.3f}")

print(f"\nand more groups tighten it -- worst case at alpha*beta={MAX_DIV}, T={T}:")
for mm in (1, 3, 8):
    print(f"  m = {mm}  ->  epsilon = {theorem4([MAX_DIV/ALPHA]*T, m=mm):.3f}")

## 8. The knob

αβ is the privacy/utility dial. The paper sweeps it over 0.01–0.10 and reports perplexity
falling from 1.459 to 1.426 while attack success rises from 0.26 to 0.29 (Table 4) — a small
range, because on TAB-ECHR the private tokens are only ~16% of the text.

The cell below re-solves λ at a *single* step across budgets, which is cheap and shows the
dial's shape directly. A real sweep regenerates the document at each setting.

In [ ]:
# The caps that actually bite depend on how far apart the two runs are at this step, so anchor
# the sweep to the measured D(lambda=1) rather than to a fixed grid that may sit entirely
# above it -- in which case every lambda is 1 and the dial looks broken when it is just slack.
d_at_1 = {g: float(compute_renyi_divergence_clipped_symmetric(p_priv[g], p_pub, ALPHA)) for g in GROUPS}
d_max = max(d_at_1.values())
print("D(lambda=1) at this step:", {g: round(d, 5) for g, d in d_at_1.items()})
print(f"-> any cap above {d_max:.5f} leaves every group unconstrained\n")

caps = sorted({round(d_max * f, 6) for f in (0.02, 0.1, 0.3, 0.6, 1.0)} | {0.01, 0.10})
print(f"{'alpha*beta':>11} | " + " | ".join(f"{g:>9}" for g in GROUPS) + " | epsilon(T=100, worst case)")
print("-" * (13 + 12 * len(GROUPS) + 27))
for ab in caps:
    lams = [find_lambda(p_priv[g], p_pub, alpha=ALPHA, beta=ab)[0] for g in GROUPS]
    eps = theorem4([ab / ALPHA] * 100, m=len(GROUPS))
    tag = "  <- paper's range" if ab in (0.01, 0.10) else ""
    print(f"{ab:11.5f} | " + " | ".join(f"{l:9.4f}" for l in lams) + f" | {eps:19.2f}{tag}")

print("\nlambda -> 0 hides the private tokens entirely; lambda -> 1 is the undefended model.")
print("The paper sweeps alpha*beta over 0.01 .. 0.10 (Table 4). Whether that range constrains")
print("anything depends on the model and the document: here the two runs agree closely enough")
print("that it often does not, which is the regime where DP-Fusion costs almost no utility.")


## 9. Scratch

`model`, `tokenizer`, `ctx`, `TEXT`, `SPANS`, `lam_hist` and `div_hist` are all live.

Things worth trying from here:

- swap `GROUPS` to `TAB_ENTITY_TYPES` for the full eight-group setup, and watch per-group ε
  fall while each group's share of the mixture shrinks to 1/8,
- set `GROUPS = ["ALL"]` with a single group covering every span — the paper's single-group
  variant (Appendix A.19), cheaper and with a smoother privacy/utility curve,
- feed spans from `fusit.trace` instead of TAB's gold annotations, which is the end-to-end
  pipeline this repo is building toward.

In [ ]:
free = lambda: (globals().pop("model", None), torch.cuda.empty_cache())
free()
print(f"GPU allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")